# colab_05 — Annotation, glia subset, and astrocyte residual-batch adjudication

Fifth notebook. Input is `integrated/scvi_integrated_full.h5ad` from colab_04 (694,922 cells /
134 donors / 46 Leiden clusters; full-gene raw counts in `.X`, scVI latent in `.obsm["X_scVI"]`).

**Three jobs:**
1. **Annotate** the 46 clusters (marker-score argmax, verified against canonical markers and the
   SEA-AD `orig_celltype` cross-check).
2. **Subset astrocytes + microglia** (ITS) from the full-gene object → the FM/FT substrate.
3. **Adjudicate the astrocyte within-niche residual-batch flag** from colab_04: cluster 22 is
   Li-pure (0.963); clusters 33/42/45 are SEA-AD-pure (~0.99). Technical batch, real-but-confounded
   biology, over-clustering, or a donor effect?

**The adjudication design is pre-registered** in `What we did.txt` (colab_05 preamble, written
before this notebook and before looking at new data). Calls baked in here:
- Stakes are **eval integrity**, not the FT substrate — CPT input is raw genes, not scVI space, and
  subsetting is robust either way. If the structure is batch/leakage it corrupts eval#2 (APOE-axis
  within astrocytes) and eval#1 (substate probe).
- **Six-lever battery, cheap-high-yield first:** (L5) donor decomposition → (L1) cross-study region
  overlap → (L4) QC-covariate + niche-mito diagnostic → (L3) DE program coherence → (L2) cross-study
  program replication → (L6) 2nd-scVI on the glia subset, **gated behind a batch verdict, run last**
  (reflexive re-integration would over-correct genuine reactive-astrocyte biology).
- Adjudicate cluster 22 (Li) and 33/42/45 (SEA-AD) **separately**; test 33/42/45 for over-clustering
  before testing for batch.
- Verdict is **weight-of-evidence, not proof** — study ≈ region ≈ sequencing-tech are near-confounded.

**Also here:** ID tiny cluster 0 (micro-adjacent, SEA-AD 0.70 — homeostatic microglia vs
perivascular macrophage?); **Audit C re-check** on the 134-donor set (SEA-AD lost 21
intermediate-ADNC donors in colab_04 5c → confirm ≥3 donors per test-stratum still holds).

**Runtime:** integration env (`requirements_integration.txt`, Py3.12, scvi-tools 1.4.3). The
2nd-scVI lever (6g, gated) needs the GPU; annotation / subset / levers L1–L5 are CPU-fine.

## 1 — Setup

### 1a — Mount Drive + clone/pull repo + install env

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info[:2]}"

# Pin numpy first so pip picks numpy-1.x-compatible wheels (same rationale as colab_01-04).
!pip install numpy==1.26.4
!pip install -r {REPO_PATH}/requirements_integration.txt

## 2 — Environment capture

### 2a — pip freeze + env JSON

In [ ]:
import json, platform, subprocess, sys
from datetime import date

NOTEBOOK_ID = "colab_05"
TODAY = date.today().isoformat()
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

def _ver(mod):
    try:
        return __import__(mod).__version__
    except Exception:
        return None

env_snapshot = {
    "notebook_id":    NOTEBOOK_ID,
    "date":           TODAY,
    "python_version": sys.version,
    "platform":       platform.platform(),
    "os_release":     platform.release(),
    "gpu":            _run(["nvidia-smi", "-L"]),
    "nvidia_driver":  _run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]),
    "git_commit":     _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
    "scvi_tools_version": _ver("scvi"),
    "scanpy_version":     _ver("scanpy"),
    "anndata_version":    _ver("anndata"),
    "jax_version":        _ver("jax"),
    "scib_metrics_version": _ver("scib_metrics"),
}
try:
    import torch
    env_snapshot["torch_version"]      = torch.__version__
    env_snapshot["torch_cuda_version"] = torch.version.cuda
    env_snapshot["cuda_available"]     = bool(torch.cuda.is_available())
    env_snapshot["cudnn_version"]      = torch.backends.cudnn.version() if torch.cuda.is_available() else None
except ImportError:
    env_snapshot["torch_version"]  = None
    env_snapshot["cuda_available"] = None
    env_snapshot["cudnn_version"]  = None

ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))

## 3 — Load the integrated object

### 3a — Load `scvi_integrated_full.h5ad`, sanity-check latent + raw counts

In [ ]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt

try:
    import psutil
    def _ram(tag):
        m = psutil.virtual_memory()
        print(f"[RAM] {tag:24s}: {m.used/1e9:5.1f} / {m.total/1e9:.1f} GB ({m.percent:.0f}%)")
except ImportError:
    def _ram(tag): pass

INTEGRATED_PATH = os.path.join(DRIVE_ROOT, "integrated", "scvi_integrated_full.h5ad")
if not os.path.exists(INTEGRATED_PATH):
    raise FileNotFoundError(f"missing integrated object {INTEGRATED_PATH} (colab_04 7a output)")

adata = sc.read_h5ad(INTEGRATED_PATH)
print("loaded:", adata.shape)
_ram("loaded integrated")

# structural sanity checks (fail loud)
assert "X_scVI" in adata.obsm, "missing obsm['X_scVI'] - colab_04 latent not present"
for col in ["study_id", "leiden_scvi", "donor_id", "region", "diagnosis",
            "apoe_carrier", "apoe_genotype"]:
    assert col in adata.obs.columns, f"missing obs['{col}']"

# raw-counts guard (same random-sample integer check as colab_04 3a; head-slice hides sorted tails)
_rng = np.random.default_rng(0)
_idx = _rng.choice(adata.n_obs, size=min(2000, adata.n_obs), replace=False)
Xs = adata.X[_idx]
_data = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
frac_int = float(np.mean(np.mod(_data, 1) == 0)) if _data.size else 1.0
assert frac_int >= 0.99, f".X not raw counts (int frac {frac_int:.3f})"

adata.obs["leiden_scvi"] = adata.obs["leiden_scvi"].astype(str)
print(f"cells={adata.n_obs}  genes={adata.n_vars}  donors={adata.obs['donor_id'].nunique()}")
print(f"leiden clusters: {adata.obs['leiden_scvi'].nunique()}  (expected 46)")
print("study fractions:\n", adata.obs["study_id"].value_counts(normalize=True).round(3))
print("raw-counts int-frac:", round(frac_int, 3))

## 4 — Annotate the 46 clusters

### 4a — Per-cluster marker DE + canonical-marker dotplot

In [ ]:
# log-normalized layer for marker visualization + scoring; .X stays raw counts throughout
adata.layers["lognorm"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4, layer="lognorm")
adata.uns.pop("log1p", None)        # stale guard from colab_04 would make layer-level log1p silently skip
sc.pp.log1p(adata, layer="lognorm")

# canonical markers per major brain cell type (only those present in the intersection gene set)
MARKERS = {
    "astrocyte":        ["AQP4", "SLC1A2", "GFAP", "GJA1", "SLC1A3"],
    "microglia":        ["CSF1R", "P2RY12", "TMEM119", "AIF1", "TREM2", "CX3CR1"],
    "oligodendrocyte":  ["MBP", "MOBP", "PLP1", "MOG"],
    "OPC":              ["PDGFRA", "CSPG4", "OLIG1", "OLIG2"],
    "neuron_exc":       ["RBFOX3", "SNAP25", "SLC17A7", "SATB2"],
    "neuron_inh":       ["GAD1", "GAD2"],
    "endothelial":      ["CLDN5", "FLT1", "PECAM1"],
    "mural":            ["PDGFRB", "RGS5", "DCN"],
    "perivascular_mac": ["MRC1", "CD163", "LYVE1", "F13A1"],
}
present = {ct: [g for g in gs if g in adata.var_names] for ct, gs in MARKERS.items()}
for ct, gs in present.items():
    missing = set(MARKERS[ct]) - set(gs)
    if missing:
        print(f"[markers] {ct}: absent from gene set -> {sorted(missing)}")

# per-cluster top DE genes for eyeballing identity
# t-test used (not Wilcoxon): this is the full 694k-cell object; Wilcoxon is O(n^2) per gene
# and timed out at >3h on A100. t-test is fast and sufficient for visual cluster annotation.
sc.tl.rank_genes_groups(adata, groupby="leiden_scvi", layer="lognorm",
                        method="t-test", n_genes=15, use_raw=False)
sc.pl.rank_genes_groups(adata, n_genes=15, sharey=False, show=True)

# dotplot of canonical markers across all clusters
flat = [g for gs in present.values() for g in gs]
sc.pl.dotplot(adata, var_names=flat, groupby="leiden_scvi", layer="lognorm",
              standard_scale="var", show=True)

### 4b — Marker-score argmax assignment + cross-check

In [ ]:
# score each cell per cell-type signature (on lognorm via an X-swap), assign each Leiden cluster
# to the highest mean-scoring type, and fail loud on low-margin (ambiguous) clusters.
score_cols = []
X_raw = adata.X
adata.X = adata.layers["lognorm"]          # score on log-norm
try:
    for ct, gs in present.items():
        if gs:
            sc.tl.score_genes(adata, gene_list=gs, score_name=f"score_{ct}", use_raw=False)
            score_cols.append(f"score_{ct}")
finally:
    adata.X = X_raw                         # ALWAYS restore raw counts

S = adata.obs.groupby("leiden_scvi")[score_cols].mean()
S.columns = [c.replace("score_", "") for c in S.columns]
top = S.idxmax(axis=1)
srt = np.sort(S.values, axis=1)
margin = pd.Series(srt[:, -1] - srt[:, -2], index=S.index)   # top minus 2nd-best
assign = pd.DataFrame({"cell_type": top, "margin": margin.round(3)})
with pd.option_context("display.max_rows", None, "display.width", 160):
    print("per-cluster mean marker scores:"); print(S.round(2))
    print("\nassignment (sorted by margin):"); print(assign.sort_values("margin"))
AMBIG = assign.index[assign["margin"] < 0.05].tolist()
if AMBIG:
    print(f"\n[AMBIGUOUS - manual review] score margin <0.05 at clusters: {AMBIG}")

# anchor clusters from the colab_04 territory map - assert score-argmax agrees, flag if not
ANCHORS = {
    "microglia":       ["1"],
    "astrocyte":       ["22", "28", "30", "31", "33", "42", "45"],
    "oligodendrocyte": ["16", "17", "18", "19", "21", "35"],
}
print("\nanchor check vs colab_04 territory map:")
for ct, cls in ANCHORS.items():
    for cl in cls:
        got = assign.loc[cl, "cell_type"] if cl in assign.index else "MISSING"
        flag = "" if got == ct else "   <-- DISAGREES, review markers in 4a"
        print(f"  cl{cl:>2}: colab_04={ct:15s} score-argmax={got}{flag}")

# map cluster -> cell_type onto cells (cluster 0 finalized in 4c; tentative here)
adata.obs["cell_type"] = adata.obs["leiden_scvi"].map(assign["cell_type"]).astype(str)
unmapped = adata.obs["cell_type"].isin(["nan", "None"]).sum()
assert unmapped == 0, f"{unmapped} cells unmapped - every cluster must receive a label"

# cross-check vs SEA-AD inherited labels (orig_celltype), where present
if "orig_celltype" in adata.obs.columns:
    print("\ncell_type x orig_celltype (SEA-AD cross-check):")
    xc = pd.crosstab(adata.obs["cell_type"], adata.obs["orig_celltype"].fillna("NA"))
    with pd.option_context("display.max_columns", None, "display.width", 200):
        print(xc)
print("\ncell_type x study:")
print(pd.crosstab(adata.obs["cell_type"], adata.obs["study_id"]))

### 4c — Identify tiny cluster 0 (homeostatic microglia vs perivascular macrophage)

In [ ]:
# cluster 0 is microglia-adjacent and SEA-AD-heavy (~0.70). Homeostatic microglia express
# P2RY12/TMEM119/CX3CR1; perivascular macrophages (PVM) express MRC1/CD163/LYVE1/F13A1.
pvm   = [g for g in ["MRC1", "CD163", "LYVE1", "F13A1"] if g in adata.var_names]
homeo = [g for g in ["P2RY12", "TMEM119", "CX3CR1"]    if g in adata.var_names]
probe = ["0", "1"]
sub = adata[adata.obs["leiden_scvi"].isin(probe)]
print("cluster sizes:", sub.obs["leiden_scvi"].value_counts().to_dict())
sc.pl.dotplot(sub, var_names={"PVM": pvm, "homeostatic microglia": homeo},
              groupby="leiden_scvi", layer="lognorm", standard_scale="var", show=True)
print(pd.crosstab(sub.obs["leiden_scvi"], sub.obs["study_id"], normalize="index").round(3))
# VERDICT recorded in _OUTPUT after reading the dotplot. If PVM, relabel cluster 0:
#   adata.obs.loc[adata.obs["leiden_scvi"]=="0", "cell_type"] = "perivascular_mac"
# (PVM are NOT part of the microglia niche -> would drop from the glia subset in 5a).

## 5 — Subset astrocytes + microglia (ITS)

### 5a — Subset from the full-gene object + save

In [ ]:
GLIA = ["astrocyte", "microglia"]
glia = adata[adata.obs["cell_type"].isin(GLIA)].copy()   # carries .X raw counts + X_scVI + lognorm
print("glia subset:", glia.shape)
print("cell_type x study (glia):")
print(pd.crosstab(glia.obs["cell_type"], glia.obs["study_id"]))

astro_obs = glia.obs.loc[glia.obs["cell_type"]=="astrocyte"].copy()
astro_obs["leiden_scvi"] = astro_obs["leiden_scvi"].astype(str)
astro_clusters = sorted(astro_obs["leiden_scvi"].unique(), key=int)
print("\nastro cluster x study (row-normalized, astro cells only):")
print(pd.crosstab(astro_obs["leiden_scvi"], astro_obs["study_id"], normalize="index")
      .loc[astro_clusters].round(3))

# free the big full-object lognorm now that the subset has its own copy
del adata.layers["lognorm"]; gc.collect(); _ram("post-subset")

if sp.issparse(glia.X) and glia.X.getformat() != "csr":
    glia.X = sp.csr_matrix(glia.X)
GLIA_DIR = os.path.join(DRIVE_ROOT, "glia_subset")
os.makedirs(GLIA_DIR, exist_ok=True)
GLIA_PATH = os.path.join(GLIA_DIR, "glia_subset_full.h5ad")
glia.write_h5ad(GLIA_PATH)
print("\nsaved glia subset ->", GLIA_PATH, f"({os.path.getsize(GLIA_PATH)/1e9:.2f} GB)")

## 6 — Astrocyte residual-batch adjudication

Pre-registered battery (full rationale in `What we did.txt`). Each lever and what it discriminates:

| Lever | Cell | Discriminates |
|---|---|---|
| L5 donor decomposition | 6b | 1-2 donors → **donor effect** (uncorrected by design); most-of-study → study-level, continue |
| L1 cross-study region overlap | 6c | Li-temporal vs SEA-MTG fail-to-mix → **batch**; SEA MTG/DLPFC split of 33/42/45 → **region** |
| L4 QC-covariate + niche-mito | 6d | study-pure cluster = low-depth/high-mito → **batch/QC** |
| L3 DE program coherence | 6e | technical genes (MT-/RPL/MALAT1/ambient) → **batch**; reactive/DAA program → **biology**; 33/42/45 near-identical → **over-clustering** |
| L2 cross-study program replication | 6f | shared tail across studies → **real (differential abundance)**; categorical absence → **batch** |
| L6 2nd-scVI on glia | 6g | dissolves → **batch**; persists → not-simple-batch — **GATED, run last** |

**Decision map** (pre-committed): technical-gene DE + QC-explained + region-match-doesn't-mix +
categorical-absence + dissolves → **BATCH** → run 6g, proceed. Coherent reactive/DAA program +
shared tail + tracks region within SEA + persists → **REAL, confounded** → do NOT 2nd-scVI; keep
structure, caveat confound, flag eval#2 leakage for colab_06. 33/42/45 near-identical → **OVER-
CLUSTERING** → merge. 1-2 donors → **DONOR**. Signals conflict → **AMBIGUOUS** → report both,
conservative eval call. Verdict is weight-of-evidence; adjudicate 22 (Li) and 33/42/45 (SEA)
separately.

### 6a — Astrocyte working subset

In [ ]:
astro = glia[glia.obs["cell_type"] == "astrocyte"].copy()
astro.obs["leiden_scvi"] = astro.obs["leiden_scvi"].astype(str)
ASTRO_CLUSTERS = sorted(astro.obs["leiden_scvi"].unique(), key=int)
LI_PURE  = ["22"]
SEA_PURE = ["33", "42", "45"]
MIXED    = [c for c in ASTRO_CLUSTERS if c not in LI_PURE + SEA_PURE]
for c in LI_PURE + SEA_PURE:
    assert c in ASTRO_CLUSTERS, f"cluster {c} not annotated astrocyte - revisit 4b before adjudicating"
print("astrocytes:", astro.shape)
print("  Li-pure  :", LI_PURE)
print("  SEA-pure :", SEA_PURE)
print("  mixed ref:", MIXED)
print("\nastro cluster x study (row-normalized):")
print(pd.crosstab(astro.obs["leiden_scvi"], astro.obs["study_id"], normalize="index")
      .loc[ASTRO_CLUSTERS].round(3))

### 6b — Lever 5: donor decomposition (cheapest; can short-circuit to donor-effect)

In [ ]:
# how many donors build each study-pure cluster, and how concentrated?
def donor_profile(cl):
    m = astro.obs["leiden_scvi"] == cl
    dc = astro.obs.loc[m, "donor_id"].value_counts()
    return {
        "n_cells":        int(m.sum()),
        "n_donors":       int((dc > 0).sum()),
        "top_donor_frac": round(float(dc.iloc[0] / dc.sum()), 3),
        "top3_donor_frac":round(float(dc.iloc[:3].sum() / dc.sum()), 3),
        "study":          astro.obs.loc[m, "study_id"].mode().iat[0],
    }
prof = pd.DataFrame({cl: donor_profile(cl) for cl in LI_PURE + SEA_PURE}).T
print(prof)
print("\ndonors per study available (astro):")
print(astro.obs.groupby("study_id")["donor_id"].nunique())
# READ: few donors / high top_donor_frac -> DONOR effect (accepted residual; batch_key=study_id,
# not donor, by design). Most-of-study donors -> study-level (region/tech) -> continue battery.

### 6c — Lever 1: cross-study region overlap

In [ ]:
# study is confounded with region but NOT perfectly:
#   Li = temporal cortex; SEA-AD = MTG (temporal) [+ DLPFC (frontal) if present];
#   Haney = SFG (frontal) + fusiform.
print("region x study (astro):")
print(pd.crosstab(astro.obs["study_id"], astro.obs["region"]))

sea_regions = astro.obs.loc[astro.obs["study_id"]=="SEA-AD", "region"].astype(str).unique().tolist()
print("\nSEA-AD regions present:", sea_regions)
HAS_DLPFC = any("DLPFC" in r.upper() for r in sea_regions)
if not HAS_DLPFC:
    print("[Lever 1] SEA-AD DLPFC arm ABSENT in loaded slice -> within-SEA region split untestable; "
          "only the Li-temporal vs SEA-MTG cross-study overlap arm is available "
          "(cl22 being Li-pure despite region-match with SEA-MTG leans batch).")
else:
    sea = astro.obs[astro.obs["study_id"]=="SEA-AD"]
    print("\nSEA-pure clusters x region (within SEA-AD; region-split -> real/confounded):")
    print(pd.crosstab(sea.loc[sea["leiden_scvi"].isin(SEA_PURE), "leiden_scvi"],
                      sea.loc[sea["leiden_scvi"].isin(SEA_PURE), "region"],
                      normalize="index").round(3))

print("\nstudy-pure cluster x region:")
m = astro.obs["leiden_scvi"].isin(LI_PURE + SEA_PURE)
print(pd.crosstab(astro.obs.loc[m, "leiden_scvi"], astro.obs.loc[m, "region"],
                  normalize="index").round(3))

### 6d — Lever 4: QC-covariate explanation + carried-forward niche-mito diagnostic

In [ ]:
# Are the study-pure clusters explained by depth / mito / doublets rather than biology?
QC_COLS = [c for c in ["total_counts", "n_genes_by_counts", "pct_counts_mt"]
           if c in astro.obs.columns]
print("per-cluster QC medians (astro):")
print(astro.obs.groupby("leiden_scvi")[QC_COLS].median().loc[ASTRO_CLUSTERS].round(2))

# carried-forward niche-mito diagnostic: among ANNOTATED astrocytes, does pct_counts_mt pile up
# against the 5% ceiling - especially in Li / Haney, which had no labels at QC time?
print("\nastrocyte pct_counts_mt by study:")
g = astro.obs.groupby("study_id")["pct_counts_mt"]
mito = pd.DataFrame({
    "mean": g.mean().round(2),
    "p95":  g.quantile(0.95).round(2),
    "frac_over_5pct": astro.obs.assign(o=astro.obs["pct_counts_mt"] > 5)
                       .groupby("study_id")["o"].mean().round(3),
})
print(mito)

fig, axes = plt.subplots(1, len(QC_COLS), figsize=(5*len(QC_COLS), 4))
for a_, col in zip(np.atleast_1d(axes), QC_COLS):
    for st in astro.obs["study_id"].unique():
        a_.hist(astro.obs.loc[astro.obs["study_id"]==st, col], bins=50, alpha=0.4,
                label=st, density=True)
    a_.set_title(col); a_.legend()
plt.tight_layout(); plt.show()

### 6e — Lever 3: DE program coherence + over-clustering check

In [ ]:
import re
# Decompose each study-pure cluster's DE vs the MIXED astro reference into NAMED nuisance panels,
# reported separately -- a coherent astrocyte program (reactive/DAA: GFAP/CD44/SERPINA3/C3/VIM;
# homeostatic: SLC1A2/GLUL) -> biology; ribo/mito, neuronal-ambient, or sex domination -> the
# specific kind of batch. One opaque tech_frac hides WHICH nuisance is driving study-purity.
RIBO_MITO_PAT = r"^(MT-|RPL|RPS|MRPL|MRPS)|^MALAT1$"
AMBIENT = {"SNAP25", "RBFOX3", "MEG3", "NRGN", "SYT1"}           # neuronal ambient / dissociation
SEX     = {"XIST", "DDX3Y", "UTY", "EIF1AY", "KDM5D", "RPS4Y1"}  # study sex-imbalance nuisance

def classify(name):
    name = str(name)
    if name in SEX:     return "sex"        # checked first: RPS4Y1 also matches ^RPS
    if name in AMBIENT: return "ambient"
    if re.match(RIBO_MITO_PAT, name): return "ribo_mito"
    return "biological"
NUISANCE = {"sex", "ambient", "ribo_mito"}  # reused by 6f to filter the replication signature

# fail-loud guard: if var_names aren't gene symbols (e.g. Ensembl IDs) or use a different mito
# convention, the regex silently matches nothing -> fractions read 0 -> every cluster looks
# biological. Refuse to trust the readout in that case.
_vn = astro.var_names.astype(str)
_n_ribo = _vn.str.match(RIBO_MITO_PAT).sum()
assert _n_ribo >= 5 and "MALAT1" in set(_vn) and _vn.str.startswith("MT-").any(), (
    f"nuisance regex matched too few genes (ribo/mito={_n_ribo}, "
    f"MALAT1 present={'MALAT1' in set(_vn)}, MT- present={_vn.str.startswith('MT-').any()}). "
    "Are var_names gene symbols, not Ensembl IDs? Refusing to trust nuisance fractions.")

astro.obs["batch_grp"] = np.where(astro.obs["leiden_scvi"].isin(MIXED), "mixed_ref",
                                  astro.obs["leiden_scvi"].astype(str))
X_raw = astro.X; astro.X = astro.layers["lognorm"]
try:
    sc.tl.rank_genes_groups(astro, groupby="batch_grp", groups=LI_PURE + SEA_PURE,
                            reference="mixed_ref", method="wilcoxon", n_genes=50, use_raw=False)
finally:
    astro.X = X_raw
for cl in LI_PURE + SEA_PURE:
    de = sc.get.rank_genes_groups_df(astro, group=cl)
    de = de[de["pvals_adj"] < 0.05]                    # drop non-significant high-LFC noise genes
    up = de.sort_values("logfoldchanges", ascending=False).head(20)
    fr = up["names"].map(classify).value_counts(normalize=True)
    print(f"\n--- cluster {cl} vs mixed_ref --- top-20 up: "
          f"ribo/mito={fr.get('ribo_mito',0):.2f}  ambient={fr.get('ambient',0):.2f}  "
          f"sex={fr.get('sex',0):.2f}  biological={fr.get('biological',0):.2f}")
    print(up[["names", "logfoldchanges", "pvals_adj"]].to_string(index=False))

# over-clustering check (cause c): are 33/42/45 three distinct programs or one state split 3 ways?
# compare each SEA-pure cluster to the SAME neutral mixed_ref - NOT to default "rest", which
# contains the sibling SEA-pure clusters and would cancel their shared program out of the top
# markers -> falsely low Jaccard -> falsely "distinct programs".
X_raw = astro.X; astro.X = astro.layers["lognorm"]
try:
    sc.tl.rank_genes_groups(astro, groupby="batch_grp", groups=SEA_PURE,
                            reference="mixed_ref", method="wilcoxon", n_genes=25, use_raw=False)
finally:
    astro.X = X_raw
def _sig_top(cl, n=20):
    d = sc.get.rank_genes_groups_df(astro, group=cl)
    d = d[d["pvals_adj"] < 0.05]
    return set(d.sort_values("logfoldchanges", ascending=False).head(n)["names"])
tops = {cl: _sig_top(cl) for cl in SEA_PURE}
print("\nSEA-pure pairwise top-20 marker Jaccard (high -> over-clustering of one state):")
for i in range(len(SEA_PURE)):
    for j in range(i+1, len(SEA_PURE)):
        a_, b_ = SEA_PURE[i], SEA_PURE[j]
        jac = len(tops[a_] & tops[b_]) / len(tops[a_] | tops[b_])
        print(f"  {a_} vs {b_}: {jac:.2f}")

### 6f — Lever 2: cross-study program replication

In [ ]:
# Build a (nuisance-filtered) signature from each study-pure cluster's top up-genes, score ALL
# astrocytes, and ask: does the program appear as a TAIL in the other studies (real substate,
# differential abundance) or is it CATEGORICALLY absent elsewhere (batch / study-unique technical)?
# Filter ribo/mito + neuronal-ambient + sex (the 6e panels) so the tested residual is the candidate
# biology, not the nuisance that already explains study-purity.
X_raw = astro.X; astro.X = astro.layers["lognorm"]
try:
    sc.tl.rank_genes_groups(astro, groupby="batch_grp", groups=LI_PURE + SEA_PURE,
                            reference="mixed_ref", method="wilcoxon", n_genes=80, use_raw=False)
    sigs = {}
    for cl in LI_PURE + SEA_PURE:
        de = sc.get.rank_genes_groups_df(astro, group=cl)
        de = de[de["pvals_adj"] < 0.05]                        # significant only (drop LFC noise)
        de = de[de["names"].map(classify) == "biological"]     # drop ribo/mito + ambient + sex
        sigs[cl] = de.sort_values("logfoldchanges", ascending=False).head(30)["names"].tolist()
    for cl, gs in sigs.items():
        sc.tl.score_genes(astro, gene_list=gs, score_name=f"prog_{cl}", use_raw=False)
finally:
    astro.X = X_raw
prog_cols = [f"prog_{cl}" for cl in LI_PURE + SEA_PURE]
# PRIMARY (non-circular): score ONLY the well-mixed reference astrocytes, split by study. The
# signature is by construction enriched in the source study's pure cluster, so a mean over ALL
# astrocytes is partly circular (the cluster inflates its own study's mean). Whether the program
# shows up in the SHARED mixed background of the OTHER studies is the real replication test.
mref = astro.obs[astro.obs["batch_grp"] == "mixed_ref"]
print("PRIMARY - program score in MIXED-REF astrocytes by study (non-circular replication test):")
print(mref.groupby("study_id")[prog_cols].mean().round(3))
print("\nsecondary - program score over ALL astrocytes by study (source study inflated by own cluster):")
print(astro.obs.groupby("study_id")[prog_cols].mean().round(3))

fig, axes = plt.subplots(1, len(prog_cols), figsize=(5*len(prog_cols), 4))
for a_, col in zip(np.atleast_1d(axes), prog_cols):
    for st in astro.obs["study_id"].unique():
        a_.hist(astro.obs.loc[astro.obs["study_id"]==st, col], bins=60, alpha=0.4,
                label=st, density=True)
    a_.set_title(col); a_.legend()
plt.tight_layout(); plt.show()

### 6g — Lever 6 (GATED): 2nd-scVI on the glia subset

**Run only on a BATCH verdict from 6b–6f.** Reflexive re-integration would over-correct genuine
reactive-astrocyte biology (decision map above). Flip `RUN_2ND_SCVI` to `True` only if the battery
points to batch. Decisive read: do 22 / 33,42,45 **dissolve** into the mixed astro manifold (batch)
or **persist** (not simple batch)?

In [ ]:
RUN_2ND_SCVI = False   # <- flip to True ONLY on a BATCH verdict from 6b-6f

if RUN_2ND_SCVI:
    import torch, scvi
    assert torch.cuda.is_available(), "2nd-scVI needs a GPU runtime"
    scvi.settings.seed = 0
    g2 = glia.copy()   # re-integrate glia ALONE: capacity focuses on glia, not the neuron majority
    sc.pp.highly_variable_genes(g2, flavor="seurat_v3", n_top_genes=3000,
                                batch_key="study_id", subset=False)
    NICHE = ["APOE", "TREM2", "MS4A6A", "CLU", "GFAP", "AQP4", "AIF1", "CSF1R"]
    for gname in NICHE:
        if gname in g2.var_names:
            g2.var.loc[gname, "highly_variable"] = True
    g2_hvg = g2[:, g2.var["highly_variable"]].copy()
    scvi.model.SCVI.setup_anndata(g2_hvg, batch_key="study_id")
    m2 = scvi.model.SCVI(g2_hvg, n_latent=30, n_layers=2, gene_likelihood="nb")
    m2.train(max_epochs=200, early_stopping=True, early_stopping_patience=15)

    glia.obsm["X_scVI_glia"] = m2.get_latent_representation(g2_hvg)
    del g2, g2_hvg; gc.collect()
    sc.pp.neighbors(glia, use_rep="X_scVI_glia", n_neighbors=15)
    sc.tl.leiden(glia, resolution=1.0, key_added="leiden_glia", flavor="igraph",
                 n_iterations=2, directed=False)
    sc.tl.umap(glia, min_dist=0.3)

    re_astro = glia[glia.obs["cell_type"] == "astrocyte"]
    print("re-integrated astro cluster x study (did 22 / 33,42,45 dissolve?):")
    print(pd.crosstab(re_astro.obs["leiden_glia"], re_astro.obs["study_id"], normalize="index").round(3))
    sc.pl.umap(re_astro, color=["study_id", "leiden_glia", "leiden_scvi"], ncols=3, show=True)

    glia.write_h5ad(GLIA_PATH)   # persist re-integrated latent for the FM notebooks
    print("re-integrated latent (X_scVI_glia) saved ->", GLIA_PATH)
else:
    print("2nd-scVI skipped - gated by design; verdict from 6b-6f was not BATCH.")

## 7 — Audit C re-check (134-donor set)

### 7a — Donor counts per (disease × APOE × study) stratum

In [ ]:
# colab_04 5c dropped 21 intermediate-ADNC donors (all SEA-AD). Re-check that the donor-level
# 70/15/15 split stratified by disease x APOE x study still leaves >=3 donors per TEST stratum.
donors = (adata.obs[["donor_id", "diagnosis", "apoe_carrier", "study_id"]]
          .drop_duplicates("donor_id"))
print("total donors:", len(donors), " by study:", donors["study_id"].value_counts().to_dict())
strata = donors.groupby(["study_id", "diagnosis", "apoe_carrier"]).size().rename("n_donors")
print("\ndonors per (study x diagnosis x APOE-carrier) stratum:")
print(strata.to_frame())

TEST_FRAC = 0.15
strata_test = np.floor(strata * TEST_FRAC).astype(int)   # floor = conservative feasibility tripwire
pinch = strata_test[strata_test < 3]
if len(pinch):
    print("\n[AUDIT C - PINCH] strata with <3 expected test donors. Locked fallback: collapse "
          "APOE -> 'any-e4', else drop the thinnest study:")
    print(pinch.to_frame("expected_test_donors"))
else:
    print("\n[AUDIT C] all strata >=3 expected test donors - split is safe.")

# Eval-facing coarse check: the per-(study x disease x APOE) granularity above guards split
# REPRESENTATIVENESS (APOE x study confound not leaking into held-out), but no eval consumes it.
# eval#2 recovers the APOE axis WITHIN astro / WITHIN micro, POOLED across study -> the binding
# constraint for eval#2 power is donors per APOE-carrier group in TEST. At this donor count the
# per-stratum check fires near-universally (all Haney strata); this coarser number is the one
# that says whether eval#2 still has legs after the fallback.
for keys in (["apoe_carrier"], ["diagnosis", "apoe_carrier"]):
    c_test = np.floor(donors.groupby(keys).size() * TEST_FRAC).astype(int)
    print(f"\n[eval#2-facing] expected TEST donors per ({' x '.join(keys)}):")
    print(c_test.to_frame("expected_test_donors"))
    thin = c_test[c_test < 3]
    if len(thin):
        print("  -> wide donor-level error bars expected for eval#2 here:", thin.index.tolist(),
              "\n     (N=3 seeds do NOT fix this -- seeds vary LoRA/RNG, the donor split is fixed)")

# secondary: a donor with ~0 astro/micro cells in test contributes nothing -- show cells available
# per (cell_type x APOE-carrier) so thin counts diluting nominal donor presence don't slip through.
if "glia" in globals():
    print("\n[eval#2-facing] glia cells per (cell_type x APOE-carrier) (full set, pre-split):")
    print(glia.obs.groupby(["cell_type", "apoe_carrier"]).size().rename("n_cells").to_frame())

## 8 — Verdict + audit trace + handoff

### 8a — Write annotation + adjudication trace; commit instructions

In [ ]:
import shlex
AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)

report["annotation"] = {
    "status": "computed",
    "date": TODAY,
    "n_cells": int(adata.n_obs),
    "cell_type_counts": adata.obs["cell_type"].value_counts().to_dict(),
    "n_astrocytes": int((adata.obs["cell_type"] == "astrocyte").sum()),
    "n_microglia":  int((adata.obs["cell_type"] == "microglia").sum()),
    "glia_subset_path": GLIA_PATH,
}
# verdict + notes filled in _OUTPUT after reading Levers 6b-6f (one of: batch / real_confounded /
# over_clustering / donor_effect / ambiguous). Pre-registered decision map in What we did.txt.
report["astro_batch_adjudication"] = {
    "status": "computed",
    "date": TODAY,
    "li_pure_clusters": LI_PURE,
    "sea_pure_clusters": SEA_PURE,
    "verdict": None,
    "ran_2nd_scvi": bool(RUN_2ND_SCVI),
    "notes": "",
}
with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print("=== Artifacts ===")
print("  committable:", FREEZE_PATH)
print("  committable:", ENV_JSON_PATH)
print("  committable:", AUDIT_REPORT_PATH)
print("  drive-only :", GLIA_PATH)
rel = [os.path.relpath(p, REPO_PATH) for p in (FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH)]
print("\n=== Commit + push (from WSL - Colab has no git creds) ===")
print("  cd /content/ad-glia-fm-prep && git add " + " ".join(shlex.quote(r) for r in rel))
print("  cd /content/ad-glia-fm-prep && git commit -m "
      "'colab_05: annotation + glia subset + astro batch adjudication trace'")
print("  cd /content/ad-glia-fm-prep && git push")

### Carried forward to colab_06

- **2nd integration method (scANVI preferred):** warm-start from the colab_04 scVI model
  (`SCANVI.from_scvi_model`) using these `cell_type` labels; run full **scIB** (batch:
  iLISI/kBET/graph connectivity; bio: NMI/ARI/ASW vs `cell_type`) comparing scVI vs scANVI.
- **APOE × study confound** (flagged colab_04 6a): quantitative cross-tab now that labels exist —
  the load-bearing reason scANVI (label-aware) is the chosen 2nd method.
- **Audit B flip:** SEA-AD + Li entries still `skipped` in `audit_report.json` → flip to `pass`.
- **Carry the astro verdict** into the eval contract: if batch/over-cluster resolved, eval#2
  (APOE within astrocytes) runs clean; if real-confounded, eval#2 must report the study/region
  confound and treat APOE recovery as confound-susceptible.